# LAB 5 — Contextual Retrieval (#T09) com Groq + Ollama BGE-M3
## Aula 4: OpenSearch — Dense + Hybrid + Neural Sparse + Contextual Retrieval
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

---

**Stack** (alinhada à revisão de 2026-05-26):
- **LLM**: Groq Cloud `llama-3.1-8b-instant` — fallback Ollama `llama3.2:3b`
- **Embeddings**: Ollama `bge-m3` (1024d) via connector ML Commons (LAB1)
- **Retrieval**: OpenSearch 3.x hybrid search (BM25 + neural kNN)
- **Avaliação**: RAGAS (Context Precision/Recall) + LangFuse (Scores API)

**Pré-requisitos:** LAB1 completo (corpus indexado e `lab1_config.json` gravado).

**Objetivo:** Comparar **Context Precision/Recall** *antes* e *depois* do Contextual Retrieval (Anthropic, 2024) sobre 100 chunks do corpus TCU 2026 e registrar a melhoria no LangFuse.

---

## Conceito

```
Chunk isolado:
  "O Tribunal, por unanimidade, negou provimento ao recurso."
  → embedding genérico, retrieval fraco

Chunk enriquecido:
  "[CONTEXTO: Acórdão TCU 2344/2026 — representação contra operação de 
   crédito BB/Município de Óbidos. Dispositivo final do voto do relator.]
   O Tribunal, por unanimidade, negou provimento ao recurso."
  → embedding rico, retrieval preciso
```

## 1. Instalação

In [ ]:
import subprocess, sys
for pkg in ['opensearch-py>=2.7', 'langchain', 'langchain-openai', 'langchain-community',
            'openai>=1.0', 'ragas>=0.1.0', 'datasets', 'pandas', 'tqdm',
            'langfuse>=2.36.0', 'python-dotenv', 'requests']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependências instaladas.')

## 2. Configurações

In [ ]:
import os, json, time, requests, warnings
from datetime import datetime
from typing import List, Optional
from tqdm.notebook import tqdm
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

# OpenSearch
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASS = os.getenv('OPENSEARCH_PASS', 'admin')

# Ollama (embeddings e LLM fallback)
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL', 'llama3.2:3b')

# Groq (LLM primário)
GROQ_API_KEY   = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL = os.getenv('GROQ_LLM_MODEL', 'llama-3.1-8b-instant')

# LangFuse
LANGFUSE_HOST       = os.getenv('LANGFUSE_HOST', 'http://localhost:3000')
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-xxxx')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-xxxx')

INDEX_BASELINE   = 'juridico_baseline_aula4'
INDEX_CONTEXTUAL = 'juridico_contextual_aula4'
EMBED_DIM = 1024

print(f'OpenSearch: {OPENSEARCH_HOST}:{OPENSEARCH_PORT}')
print(f'Ollama    : {OLLAMA_BASE_URL} (embed={OLLAMA_EMBED_MODEL}, llm fallback={OLLAMA_LLM_MODEL})')
print(f'Groq      : modelo={GROQ_LLM_MODEL} (key set? {bool(GROQ_API_KEY)})')
print(f'LangFuse  : {LANGFUSE_HOST}')

## 3. Conexões: OpenSearch + LLM (Groq → Ollama)

In [ ]:
from opensearchpy import OpenSearch
from openai import OpenAI

os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False,
)
info = os_client.info()
print(f"OpenSearch {info['version']['number']} OK")
assert info['version']['number'].startswith('3'), 'Requer OpenSearch 3.x'

# Selecionar LLM: Groq primário, Ollama fallback
llm_client: Optional[OpenAI] = None
MODEL_NAME, LLM_PROVIDER = None, None
if GROQ_API_KEY:
    try:
        c = OpenAI(base_url='https://api.groq.com/openai/v1', api_key=GROQ_API_KEY)
        c.chat.completions.create(model=GROQ_LLM_MODEL,
                                   messages=[{'role':'user','content':'ok'}], max_tokens=2)
        llm_client, MODEL_NAME, LLM_PROVIDER = c, GROQ_LLM_MODEL, 'groq'
    except Exception as e:
        print(f'Groq indisponível ({e}). Fallback Ollama.')
if llm_client is None:
    llm_client = OpenAI(base_url=f'{OLLAMA_BASE_URL}/v1', api_key='ollama')
    MODEL_NAME, LLM_PROVIDER = OLLAMA_LLM_MODEL, 'ollama'

print(f'LLM ativo: {LLM_PROVIDER} ({MODEL_NAME})')

def ollama_embed(text: str) -> List[float]:
    try:
        r = requests.post(f'{OLLAMA_BASE_URL}/api/embeddings',
                          json={'model': OLLAMA_EMBED_MODEL, 'prompt': text}, timeout=60)
        r.raise_for_status()
        return r.json()['embedding']
    except Exception as e:
        print(f'Embeddings indisponíveis: {e}')
        return [0.0] * EMBED_DIM

print(f'Embed dim (smoke): {len(ollama_embed("teste"))}')

## 4. Carregar Corpus (TCU 2026) e Gerar 100 Chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

CORPUS_PATH = '../datasets/corpus_juridico_aula4_v2.json'
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus = json.load(f)
print(f'Corpus: {len(corpus)} acórdãos')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64,
    separators=['\n\n', '\n', '. ', ' ']
)

chunks_raw = []
for d in corpus:
    pieces = splitter.split_text(d.get('conteudo', ''))
    for i, p in enumerate(pieces):
        chunks_raw.append({
            'chunk_id':    f"{d['id']}_c{i:03d}",
            'doc_id':      d['id'],
            'doc_tipo':    d.get('tipo', ''),
            'doc_titulo':  d.get('titulo', ''),
            'doc_completo': d.get('conteudo', '')[:3000],
            'conteudo':    p,
            'metadata':    d.get('metadata', {}),
        })
        if len(chunks_raw) >= 100:
            break
    if len(chunks_raw) >= 100:
        break

print(f'Chunks: {len(chunks_raw)}')
print('Exemplo chunk:')
print(f"  id: {chunks_raw[0]['chunk_id']} | doc: {chunks_raw[0]['doc_id']}")
print(f"  trecho: {chunks_raw[0]['conteudo'][:160]}...")

## 5. Índices Baseline + Contextual

Como cada chunk será embedded **server-side**, **mantemos o `default_pipeline` do LAB1** (`pipeline_bge_m3_ingest`) para que o OpenSearch chame o Ollama via connector durante a ingestão.

In [ ]:
with open('lab1_config.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)
DENSE_MODEL_ID   = cfg['model_id']
INGEST_PIPELINE  = cfg['ingest_pipeline']
print(f'Pipeline para embeddings: {INGEST_PIPELINE} (model_id {DENSE_MODEL_ID})')

def mapping_chunks(default_pipeline: str):
    return {
        'settings': {
            'index': {
                'knn': True,
                'knn.algo_param.ef_search': 100,
                'default_pipeline': default_pipeline
            }
        },
        'mappings': {
            'properties': {
                'chunk_id':   {'type': 'keyword'},
                'doc_id':     {'type': 'keyword'},
                'doc_tipo':   {'type': 'keyword'},
                'doc_titulo': {'type': 'text', 'analyzer': 'portuguese'},
                'conteudo':   {'type': 'text', 'analyzer': 'portuguese'},
                'contexto_gerado': {'type': 'text', 'analyzer': 'portuguese'},
                'metadata':   {'type': 'object'},
                'knn_vector': {
                    'type': 'knn_vector',
                    'dimension': EMBED_DIM,
                    'method': {'name': 'hnsw', 'space_type': 'cosinesimil',
                               'engine': 'faiss',
                               'parameters': {'ef_construction': 128, 'm': 16}}
                }
            }
        }
    }

def reset_index(name: str, body: dict):
    if os_client.indices.exists(index=name):
        os_client.indices.delete(index=name)
    os_client.indices.create(index=name, body=body)

reset_index(INDEX_BASELINE,   mapping_chunks(INGEST_PIPELINE))
reset_index(INDEX_CONTEXTUAL, mapping_chunks(INGEST_PIPELINE))
print(f'Índices criados: {INDEX_BASELINE}, {INDEX_CONTEXTUAL}')

## 6. Indexar Baseline (chunks crus)

In [ ]:
def bulk_index(index: str, items: list, campo_texto: str = 'conteudo'):
    bulk = []
    for it in items:
        bulk.append({'index': {'_index': index, '_id': it['chunk_id']}})
        body = {k: v for k, v in it.items() if k != 'doc_completo'}
        # garantir que o campo do qual o pipeline gera embedding está presente
        body['conteudo'] = it[campo_texto]
        bulk.append(body)
    os_client.bulk(body=bulk, request_timeout=180)
    os_client.indices.refresh(index=index)

bulk_index(INDEX_BASELINE, chunks_raw, campo_texto='conteudo')
print(f"baseline docs: {os_client.count(index=INDEX_BASELINE)['count']}")

## 7. Contextual Retrieval — Enriquecer 100 chunks via Groq/Ollama

In [ ]:
PROMPT_SYS = ('Você é um analista jurídico especializado em direito brasileiro e em acórdãos do TCU.')

def gerar_contexto(doc_titulo: str, doc_completo: str, chunk: str) -> str:
    msg = (
        f'Documento (início):\n{doc_completo}\n\n'
        f'Título do documento: {doc_titulo}\n\n'
        f'Trecho a contextualizar:\n{chunk}\n\n'
        'Gere UMA frase (máx. 40 palavras) descrevendo o papel deste trecho no documento. '
        'Inclua: tipo do documento, número/ano (se houver), partes/órgão, tema e posição. '
        'Responda SOMENTE a frase.'
    )
    try:
        r = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{'role': 'system', 'content': PROMPT_SYS},
                      {'role': 'user',   'content': msg}],
            temperature=0.0, max_tokens=120,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        print(f'⚠️ contexto vazio ({e})')
        return '[contexto indisponível]'

print(f'Enriquecendo {len(chunks_raw)} chunks via {LLM_PROVIDER} ({MODEL_NAME})...')
t0 = time.time()
chunks_ctx = []
for i, c in enumerate(tqdm(chunks_raw)):
    ctx = gerar_contexto(c['doc_titulo'], c['doc_completo'], c['conteudo'])
    enriquecido = c.copy()
    enriquecido['contexto_gerado']     = ctx
    enriquecido['conteudo_contextual'] = f'[CONTEXTO: {ctx}]\n\n{c["conteudo"]}'
    chunks_ctx.append(enriquecido)
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(chunks_raw)} ({time.time()-t0:.0f}s) — ex: {ctx[:80]}...')

tempo_enriquecimento = time.time() - t0
print(f'\nConcluído em {tempo_enriquecimento:.0f}s ({tempo_enriquecimento/len(chunks_raw):.2f}s/chunk)')

## 8. Indexar Contextual (chunks enriquecidos)

In [ ]:
# Indexa usando o conteúdo_contextual como o campo do qual o pipeline gera embedding.
bulk_index(INDEX_CONTEXTUAL, chunks_ctx, campo_texto='conteudo_contextual')
print(f"contextual docs: {os_client.count(index=INDEX_CONTEXTUAL)['count']}")
amostra = os_client.get(index=INDEX_CONTEXTUAL, id=chunks_ctx[0]['chunk_id'])
print(f"  amostra chunk_id  : {amostra['_source']['chunk_id']}")
print(f"  contexto gerado   : {amostra['_source']['contexto_gerado'][:120]}")
print(f"  conteudo (head)   : {amostra['_source']['conteudo'][:160]}")

## 9. Buscas Hibridas Baseline vs Contextual

In [ ]:
PIPELINE_RRF = 'hybrid_rrf_pipeline'  # criado no LAB2
K = 5

def buscar_contextos(query: str, indice: str, k: int = K) -> list[str]:
    body = {
        'size': k,
        'query': {
            'hybrid': {
                'queries': [
                    {'match':  {'conteudo': {'query': query}}},
                    {'neural': {'knn_vector': {'query_text': query, 'model_id': DENSE_MODEL_ID, 'k': k}}}
                ]
            }
        },
        '_source': ['conteudo', 'contexto_gerado']
    }
    r = os_client.search(index=indice, body=body, params={'search_pipeline': PIPELINE_RRF})
    return [h['_source']['conteudo'] for h in r['hits']['hits']]

# Queries de avaliação com ground-truth textual (para Context Precision/Recall do RAGAS)
QUERIES_RAGAS = [
    {'question':     'Quais os requisitos para operação de crédito com garantia da União?',
     'ground_truth': 'A operação de crédito com garantia da União exige autorização legislativa específica e observância da Lei de Responsabilidade Fiscal, com limites e condições estabelecidos pelo Senado e pelo TCU.'},
    {'question':     'Como se caracteriza dano ao erário em operação de crédito?',
     'ground_truth': 'Dano ao erário em operação de crédito ocorre quando há prejuízo financeiro à União decorrente de execução irregular, triangulação bancária, fraude em medições ou descumprimento contratual.'},
    {'question':     'Quais documentos devem instruir uma representação ao TCU?',
     'ground_truth': 'A representação ao TCU deve conter qualificação do representante, descrição do fato, indicação dos responsáveis, fundamentos jurídicos e documentos probatórios mínimos.'},
    {'question':     'Quando cabe medida cautelar em sede de representação?',
     'ground_truth': 'Medida cautelar é cabível quando há fundado receio de grave lesão ao erário ou a direito alheio, ou risco de ineficácia da decisão final do TCU.'},
    {'question':     'Como é apurada responsabilidade solidária por dano ao erário?',
     'ground_truth': 'A responsabilidade solidária é apurada em tomada de contas especial mediante contraditório, demonstração do nexo causal e do dolo ou culpa dos envolvidos.'}
]

print('Coletando contextos baseline vs contextual...')
for q in QUERIES_RAGAS:
    q['ctx_baseline']   = buscar_contextos(q['question'], INDEX_BASELINE,  k=K)
    q['ctx_contextual'] = buscar_contextos(q['question'], INDEX_CONTEXTUAL, k=K)
    print(f"  {q['question'][:60]}... → {len(q['ctx_baseline'])} | {len(q['ctx_contextual'])}")

## 10. RAGAS: Context Precision/Recall — Antes vs Depois

Configuramos o RAGAS para usar o **LLM ativo** (Groq/Ollama via OpenAI-compatible) e **embeddings Ollama BGE-M3**.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import OllamaEmbeddings

# Wrapper RAGAS — usa endpoint OpenAI-compatible (Groq ou Ollama)
if LLM_PROVIDER == 'groq':
    ragas_llm = ChatOpenAI(
        model=GROQ_LLM_MODEL,
        base_url='https://api.groq.com/openai/v1',
        api_key=GROQ_API_KEY,
        temperature=0.0
    )
else:
    ragas_llm = ChatOpenAI(
        model=OLLAMA_LLM_MODEL,
        base_url=f'{OLLAMA_BASE_URL}/v1',
        api_key='ollama',
        temperature=0.0
    )
ragas_emb = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)

def make_ds(key_ctx: str):
    return Dataset.from_list([
        {'question': q['question'],
         'contexts': q[key_ctx],
         'ground_truth': q['ground_truth'],
         'answer': '[avaliação de retrieval]'} for q in QUERIES_RAGAS
    ])

print('Avaliando RAGAS — baseline...')
res_base = evaluate(make_ds('ctx_baseline'),
                    metrics=[context_precision, context_recall],
                    llm=ragas_llm, embeddings=ragas_emb)
print('Avaliando RAGAS — contextual...')
res_ctx  = evaluate(make_ds('ctx_contextual'),
                    metrics=[context_precision, context_recall],
                    llm=ragas_llm, embeddings=ragas_emb)

cp_base  = float(res_base['context_precision']); cp_ctx = float(res_ctx['context_precision'])
cr_base  = float(res_base['context_recall']);     cr_ctx = float(res_ctx['context_recall'])
d_cp = cp_ctx - cp_base; d_cr = cr_ctx - cr_base
print('\n=== RESULTADOS RAGAS — Baseline vs Contextual ===')
print(f'  Context Precision: {cp_base:.4f} → {cp_ctx:.4f}  (Δ {d_cp:+.4f})')
print(f'  Context Recall   : {cr_base:.4f} → {cr_ctx:.4f}  (Δ {d_cr:+.4f})')

## 11. Registrar no LangFuse

In [ ]:
from langfuse import Langfuse
try:
    lf = Langfuse(public_key=LANGFUSE_PUBLIC_KEY,
                  secret_key=LANGFUSE_SECRET_KEY,
                  host=LANGFUSE_HOST)
    trace = lf.trace(
        name='Aula4-LAB5-Contextual-Retrieval',
        metadata={
            'aula': '4', 'lab': '5', 'tecnica': '#T09 Contextual Retrieval',
            'n_chunks': len(chunks_raw),
            'embed_model': OLLAMA_EMBED_MODEL, 'embed_dim': EMBED_DIM,
            'llm_provider': LLM_PROVIDER, 'llm_model': MODEL_NAME,
            'opensearch_version': '3.x',
            'tempo_enriquecimento_s': round(tempo_enriquecimento, 1),
        }
    )
    sp_b = trace.span(name='ragas-baseline',
                       metadata={'indice': INDEX_BASELINE})
    sp_b.score(name='context_precision', value=cp_base)
    sp_b.score(name='context_recall',    value=cr_base)
    sp_b.end()
    sp_c = trace.span(name='ragas-contextual',
                       metadata={'indice': INDEX_CONTEXTUAL})
    sp_c.score(name='context_precision', value=cp_ctx)
    sp_c.score(name='context_recall',    value=cr_ctx)
    sp_c.end()
    trace.score(name='delta_context_precision', value=d_cp,
                comment='Δ Context Precision (contextual − baseline)')
    trace.score(name='delta_context_recall',    value=d_cr,
                comment='Δ Context Recall (contextual − baseline)')
    lf.flush()
    print(f'Trace: {LANGFUSE_HOST}/traces/{trace.id}')
except Exception as e:
    print(f'LangFuse indisponível: {e}')

## 12. Análise Qualitativa + Export

In [ ]:
import pandas as pd
exemplo = QUERIES_RAGAS[0]
print(f"Query: {exemplo['question']}\n")
print('── BASELINE ──')
for i, c in enumerate(exemplo['ctx_baseline'][:3], 1):
    print(f'  [{i}] {c[:180]}...')
print('\n── CONTEXTUAL ──')
for i, c in enumerate(exemplo['ctx_contextual'][:3], 1):
    print(f'  [{i}] {c[:180]}...')

df = pd.DataFrame([
    {'Estratégia': 'Baseline (sem contexto)',
     'Context Precision': f'{cp_base:.4f}', 'Context Recall': f'{cr_base:.4f}',
     'Tempo ingestão':    '~2 min / 100 chunks'},
    {'Estratégia': 'Contextual Retrieval (#T09)',
     'Context Precision': f'{cp_ctx:.4f}', 'Context Recall': f'{cr_ctx:.4f}',
     'Tempo ingestão':    f'~{tempo_enriquecimento:.0f}s / 100 chunks ({LLM_PROVIDER})'}
])
df.to_csv('lab5_contextual_retrieval.csv', index=False)
print('\nTabela comparativa:')
print(df.to_string(index=False))
print('\nlab5_contextual_retrieval.csv gravado.')

# Persistir config para LAB6
with open('lab5_config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'index_baseline':   INDEX_BASELINE,
        'index_contextual': INDEX_CONTEXTUAL,
        'cp_baseline':      cp_base, 'cp_contextual': cp_ctx,
        'cr_baseline':      cr_base, 'cr_contextual': cr_ctx,
        'llm_provider':     LLM_PROVIDER, 'llm_model': MODEL_NAME,
        'tempo_enriquecimento_s': round(tempo_enriquecimento, 1)
    }, f, ensure_ascii=False, indent=2)
print('lab5_config.json gravado.')

## Checklist

| # | Item |
|---|------|
| 1 | OpenSearch 3.x conectado |
| 2 | LLM ativo (Groq ou Ollama) confirmado |
| 3 | 100 chunks gerados e indexados no índice baseline |
| 4 | Enriquecimento contextual concluído |
| 5 | Índice contextual indexado com embeddings server-side (Ollama BGE-M3) |
| 6 | RAGAS executado com Groq/Ollama + Ollama embeddings |
| 7 | Delta Context Precision ≥ 0 e registrado no LangFuse |
| 8 | Configurações exportadas para o LAB6 |

## Referências (ABNT)

ANTHROPIC. **Introducing Contextual Retrieval**. Blog, 2024. <https://www.anthropic.com/news/contextual-retrieval>.

ES, S. et al. **RAGAS: Automated Evaluation of Retrieval Augmented Generation**. arXiv:2309.15217, 2023.

OPENSEARCH PROJECT. **Hybrid Search**. 3.0 Docs. <https://docs.opensearch.org/3.0/vector-search/ai-search/hybrid-search/>.

GROQ. **API documentation**. <https://console.groq.com/docs>.